## global packege we will need

###  Installing watchexec on Ubuntu (WSL) — Conclusion

####  What we were trying to do
Get a file-watcher that auto-restarts `cargo run` on every source change, for faster Axum API development.

#### What didn't work, and why

| Attempt | Result | Why it failed |
|---|---|---|
| `apt install watchexec` | `E: Unable to locate package` | `watchexec` isn't in the default Ubuntu/Debian apt repos |
| `cargo add watchexec --dev` | Added 60+ transitive crates to `Cargo.toml` | `cargo add` installs a **library crate** for use inside your code (`use watchexec::...`) — not a CLI binary. Wrong tool for the job. |
| Guessed GitHub release filename + `tar -xf` | `tar: This does not look like a tar archive` | The exact version/filename was guessed instead of fetched — likely downloaded a 404/HTML page instead of the real asset |
| First `grep` for the download URL | Returned 4 URLs instead of 1 | Grep pattern matched `.tar.xz`, `.tar.xz.b3`, `.tar.xz.sha256`, `.tar.xz.sha512` all at once — needed to anchor the match to end exactly at the closing quote |

#### What worked

**1. Fetch the real, current download URL from GitHub's API** (never hand-guess a version number or filename):

```bash
curl -s https://api.github.com/repos/watchexec/watchexec/releases/latest \
  | grep '"browser_download_url".*x86_64-unknown-linux-musl.tar.xz"' \
  | cut -d '"' -f 4
```

**2. Download it:**

```bash
curl -LO $(curl -s https://api.github.com/repos/watchexec/watchexec/releases/latest \
  | grep '"browser_download_url".*x86_64-unknown-linux-musl.tar.xz"' \
  | cut -d '"' -f 4)
```

**3. Verify before extracting** (this step is what would've caught the earlier failure immediately):

```bash
file watchexec-*.tar.xz
# should say: XZ compressed data
```

**4. Extract and install to PATH:**

```bash
tar -xf watchexec-*.tar.xz
sudo mv watchexec-*/watchexec /usr/local/bin/
```

**5. Confirm:**

```bash
watchexec --version
```

**6. Clean up download artifacts:**

```bash
rm -rf watchexec-*.tar.xz watchexec-*/
```

**7. Remove the earlier mistaken dev-dependency:**

```bash
cargo remove watchexec --dev
```

#### Key lessons

- **`cargo add`** = adds a dependency your Rust *code* imports (`use crate_name::...`).
- **`cargo install`** = compiles and installs a standalone CLI *binary* to `~/.cargo/bin/` — the alternative path we didn't need in the end, but works reliably too, since it always matches your exact system (no arch/libc guessing).
- **Prebuilt binaries**: always resolve the download URL dynamically via the GitHub releases API instead of guessing a version string — release filenames change over time and guesses silently 404.
- **`file <archive>`** before `tar -xf` is a cheap sanity check that saves debugging time — confirms you actually downloaded an archive, not an error page.
- For WSL/Linux x86_64, the **musl** build is a safe choice — statically linked, no glibc version compatibility concerns.

#### Final working command (daily use)

```bash
watchexec -e rs,toml -r -- cargo run
```

| Flag | Meaning |
|---|---|
| `-e rs,toml` | only restart on `.rs` or `Cargo.toml` changes |
| `-r` | restart the process (kill + rerun) instead of letting it pile up |
| `-- cargo run` | the command to actually run |

#### Optional: shortcut alias

Add to `~/.zshrc`:

```bash
alias dev="watchexec -e rs,toml -r -- cargo run"
```

```bash
source ~/.zshrc
```

Then just run `dev` from the project root to start the auto-reloading Axum server.

### global packege 

#### install cargo watch
`cargo install cargo-watch`

##### how to use it
`cargo watch -x run`


#### to track changes like nodemon
`cargo install watchexec-cli`

#### in project root folder
`watchexec -e rs -r -- cargo run`
`watchexec -e rs,toml -r -- cargo run`

Flag	Meaning
-e rs	only trigger on .rs file changes (ignores target/, Cargo.lock noise, etc.)
-r	restart the process if it's still running (kills old cargo run, starts fresh)
--	everything after this is the command to run

Even simpler — save it as an alias

Add to your ~/.zshrc (since you're on zsh):

bash
alias dev="watchexec -e rs -r -- cargo run"
alias dev="watchexec -e rs,toml -r -- cargo run"

Then reload:

bash
source ~/.zshrc

From now on, just cd into any Rust project and run:

bash
dev


# necessary Understand Before Start

## Create project and install necessary packege

### create rust project
`cargo new axum_api`

### install packege ( crates )

`cargo add axum`

`cargo add tokio --features full`

`cargo add serde --features derive`

`cargo add serde_json`

##### 1. What exactly is a Crate?

In Rust, a **crate** is the unit of compilation — what other languages would call a "package" or "library." Every Rust project you build is itself a crate, and every dependency you pull in from crates.io is also a crate.

There are two kinds:

- **Binary crate** — has a `main()` function and compiles to an executable. Your `axum_rust_api_handbook` project is a binary crate.
- **Library crate** — has no `main()`. It exposes functions, types, and traits for other crates to use. `axum`, `tokio`, and `serde` are all library crates.

When you run `cargo add axum`, Cargo downloads the `axum` library crate from crates.io, records the version in `Cargo.toml`, and locks the exact resolved version tree in `Cargo.lock`. Your binary crate then links against it at compile time — this is how a single line like `use axum::Router;` gives you access to code someone else wrote and published.

> **Crate vs Module** — don't confuse these. A crate is the whole compiled package. A **module** (`mod`) is how you organize code *within* a crate — like folders inside the package. One crate can contain many modules.

---

##### 2. Which crates your project actually uses

| Crate | Role |
|---|---|
| `axum` | The web framework itself — routing, extractors (`Path`, `Json`), request/response handling. Built on top of `hyper` + `tower`. |
| `tokio` | The async runtime. Rust has no built-in async executor — tokio is what actually drives your `async fn`s and handles the event loop, sockets, and scheduling. |
| `serde` | The serialization/deserialization framework. The `derive` feature auto-generates code so your structs can convert to/from JSON, YAML, etc. |
| `serde_json` | The JSON backend for serde. Axum's `Json<T>` extractor uses this under the hood to parse request bodies and format responses. |

> Notice the relationship: **serde** defines the general "how to serialize" traits, while **serde_json** is one specific implementation (JSON). If you swapped to YAML later, you'd keep `serde` and add `serde_yaml` instead — your struct derives wouldn't change.

---